# Discover and download Spitzer/IRAC BCD observations

Install `python -m pip install -e ".[download]"` from the SPCA repository first.
This notebook uses shared functions in `SPCA.spitzer_download`; it never rewrites FITS contents.
**Run All does not download FITS data.** Select AORs and explicitly enable downloading below.

Discovery uses IRSA SIA v2 (`Irsa.query_sia`, collection `spitzer_sha`), which lists
individual archive files; each file is downloaded directly from the IRSA SHA archive.

**Limitations:** SIA has no per-AOR file inventory, so archive completeness remains
**unverified** per AOR/channel even when every planned transfer succeeds. Only files whose
footprints cover the target are found; channels 1/3 and 2/4 view adjacent fields, so a
full-array AOR usually yields just the on-target channel pair. Section 7 can compare the
plan with the archive's own AOR directory listing and quality report. Errors are reported
as errors, never as empty successful searches.

## 1. Set target, output directory, and selections

Keep all settings in this cell. Coordinates bypass name resolution completely (important
during SIMBAD/resolver outages). The target name is only a label when coordinates are supplied.
Use a different output directory for each independently analyzed observing campaign.
Optional PID filters intersect with the positional search; a program can contain other targets.

In [ ]:
from pathlib import Path

import astropy.units as u
from astropy.coordinates import SkyCoord

from SPCA.directory_structure import make_directory_structure
from SPCA.spitzer_download import (
    LIMITATION,
    compare_inventory,
    discover_observations,
    download_plan,
    inventory_summary,
    observation_summary,
    plan_download,
    save_plan,
)

TARGET = "WASP-12b"
POSITION = SkyCoord("06h30m32.797s +29d40m20.26s", frame="icrs")
RADIUS = 0.5 * u.arcmin  # SIA matches any file whose footprint overlaps this circle.
OUTPUT = Path.home() / "Spitzer_data" / "WASP-12b_selected_campaign"
PIDS = (
    None  # Candidate filters: [50517, 50759, 70060, 90186]; not verified assignments.
)
AOR_FILTER = None  # Optional specific AORKEY list for discovery.
CHANNELS = [1, 2, 3, 4]  # 3.6, 4.5, 5.8, 8.0 microns.
# Default: the 2008 cryogenic campaign. Set both to None to search the whole mission
# (slow: IRSA SIA queries over many years can take an hour for well-observed targets).
START = "2008-01-01"  # UTC.
END = "2008-12-31T23:59:59.999"  # Inclusive UTC timestamp; a date alone means midnight.
EXPANSION_RADIUS = 0.5 * u.arcmin  # Re-query radius covering the target for each AOR.
SELECTED_AORS = []  # Copy verified AORKEYs from the results; no implicit selection of all.
SELECTED_CHANNELS = [1, 2, 3, 4]
PROFILE = "spca"  # Default. "science-qa" adds *_bunc/*_bimsk; "archive" keeps all.
SCIENCE_PRODUCT = "bcd"  # Or "cbcd"; never substituted for one another.
COMPARE_WITH_ARCHIVE = (
    False  # True: check the plan against IRSA AOR listings (metadata).
)
SHA_REFERENCE = None  # Or SHA ZIP file(s)/extracted directories to compare against.
DOWNLOAD = False  # Set True only after reviewing selected AORs and the preview.

print(LIMITATION)

## 2. Discover relevant observations

A positional SIA query identifies AORs; SIA matches every file whose image footprint
overlaps the search circle, so frames dithered or mapped around the target are found.
Each AOR is then re-queried (`EXPANSION_RADIUS`) over that AOR's full time span, so all of
its files covering the target are gathered even when a date filter catches only part of a
long AOR. The span comes from the archive's AOR-level records or quality report when
available (`time_coverage` column); otherwise the search grows until no further frames are
found and a warning is shown. Frames that never cover the target (channels 1/3 and 2/4
view adjacent fields) are not retrieved.
Products are classified by
their archive path (`bcd/`, `cal/`, `raw/`, `pbcd/`), not by IVOA calibration level.
SIA cannot search by PID/AOR, so coordinates are always required; PIDs/AORs then filter
the positional results.
IRSA silently truncates large SIA results, so long date ranges are split automatically
into shorter queries; for WASP-12b the 2008 search below took about 25 minutes.
Do not interpret a failed API request as evidence that observations do not exist.

In [ ]:
observations = discover_observations(
    position=POSITION,
    target=TARGET,
    radius=RADIUS,
    pids=PIDS,
    aorkeys=AOR_FILTER,
    channels=CHANNELS,
    start=START,
    end=END,
    expansion_radius=EXPANSION_RADIUS,
)

## 3. Inspect the candidate observation summary

One row describes each discovered AOR/channel. Missing PID/date/mode values remain blank.
AOR assignments and 2008 channel availability must be read from actual returned metadata,
not assumed from the candidate program list. Discovery is independent of `PROFILE`:
`science_exposures` counts unique exposures (a BCD and its CBCD are one exposure),
`bcd_files`/`cbcd_files` show which versions exist, and `archive_bytes` sums all returned
Level 1 files. These describe returned archive metadata, not authoritative archive totals;
the preview below gives the size for the chosen profile. `excluded_files` counts Level 0
(`raw/`) and Level 2 (`pbcd/`) products that are never downloaded.

In [ ]:
display(observations)
print("Candidate AORs:", sorted(set(observations["aorkey"])))

## 4. Select AORs/channels

Set `SELECTED_AORS` and `SELECTED_CHANNELS` in the settings cell and rerun it before continuing.
Select a single campaign for a light curve. Selection never merges AORs or epochs automatically.
Each selected AOR is re-queried over the AOR's full time span (see `time_coverage`),
retaining the whole AOR rather than a date-clipped subset.

In [ ]:
plan = None
if SELECTED_AORS:
    plan = plan_download(
        observations,
        OUTPUT,
        aorkeys=SELECTED_AORS,
        channels=SELECTED_CHANNELS,
        profile=PROFILE,
        science_product=SCIENCE_PRODUCT,
    )
else:
    print("No AORs selected. Edit SELECTED_AORS above; no download is planned.")

## 5. Preview files and size

Sizes are archive estimates (SIA `access_estsize`) or exact sizes where provided; files of
unknown size are counted separately, never as zero. Preview writes only a small ECSV plan.
Without authoritative archive counts, completeness stays unverified for each AOR/channel.

In [ ]:
if plan is not None:
    summary = observation_summary(plan)
    display(summary)
    display(plan)
    print(f"Estimated download size: {sum(summary['estimated_bytes']) / 1e9:.3f} GB")
    print("Files of unknown size:", sum(summary["unknown_sizes"]))
    print("Not planned (by reason):", plan.meta["excluded_products"])
    print("AOR time coverage:", plan.meta["time_coverage"])
    print("Saved plan:", save_plan(plan))

### Optional: compare the plan with an independent file listing

SIA is a positional search, so frames that never cover the target (e.g. off-target maps,
the adjacent-field channel pair) are not planned. `compare_inventory` checks the planned
`bcd/` filenames per AOR/channel against either the IRSA archive's own directory listing and
`QualityAnalysis` report (`COMPARE_WITH_ARCHIVE = True`; a few small metadata requests) or
SHA ZIP files/extracted directories you downloaded independently (`SHA_REFERENCE`).
It reports missing, extra and intentionally excluded files, plus channels holding data that
were not planned. The archive listing is an HTML index, not a versioned API.

In [ ]:
comparison = None
if plan is not None and (COMPARE_WITH_ARCHIVE or SHA_REFERENCE):
    comparison = compare_inventory(plan, SHA_REFERENCE)
    display(comparison)
    print(
        "Channels with data not in the plan:",
        comparison.meta["unplanned_channels_with_data"],
    )
    for key, detail in comparison.meta["details"].items():
        print(key, "missing:", detail["missing"], "extra:", detail["extra"])

## 6. Explicitly download

Only set `DOWNLOAD=True` after reviewing the preview. Streaming transfers use temporary
files, bounded retries/timeouts, size checks when provided, and atomic completion.
Rerunning rechecks local SHA-256 hashes and skips verified completed files; interrupted
files restart from the beginning. Local hashes are not archive-supplied checksums.
Each FITS file is downloaded individually into `data/chN/r<AOR>/chN/<bcd|cal>/`, with one
overall progress bar. The manifest and inventory are checkpointed periodically (every 100
files or 30 s) and on completion, failure or interruption.

In [ ]:
inventory = None
if DOWNLOAD and plan is not None:
    inventory = download_plan(plan)
elif DOWNLOAD:
    raise ValueError("Select AORs and create a plan before enabling downloads")
else:
    print("Download disabled. No FITS files transferred.")

## 7. Check the inventory and prepare SPCA

`download_manifest.ecsv` records transfer statuses/errors and original URLs;
`download_inventory.ecsv` records actual file paths, sizes, product categories, and local hashes.
Science/uncertainty/mask counts describe downloaded files, **not authoritative archive totals**.
Equal counts do not establish completeness; use `compare_inventory(inventory, ...)` (above)
for an independent check. The downloader preserves both full-frame 2D and
subarray 3D FITS bytes; it does not flatten cubes or run photometry.

Directory setup reuses `data/chN/r########/chN/bcd/` and creates the standard analysis folders.
Next configure `Everything_Photometry.py` or its notebook with this campaign root and the
desired channel/AOR selection. Existing SHA ZIP downloads remain supported by
`Make_Directory_Structure`. Never combine separate campaigns merely because their AORs
share a prefix.

In [ ]:
if inventory is not None:
    display(inventory_summary(inventory))
    display(inventory)
    snippet = make_directory_structure(OUTPUT)
    print("Campaign root:", OUTPUT)
    print("AOR snippet:", snippet)
    print("Check completeness limitations before starting photometry.")